### Project Description
- Collect Weekly Data for $p = 9$ stocks over period $n = 26$ weeks from SP500 Index
- Compute weekly excess returns matrix
- subtract mean from each row to obtain de-meaned excess returns matrix $Y$
- compue sample covariance matrix $YY^T/n$
- compue global-minimum-variance portfolio holding vector $h_C$
- compute expected excess return, variance, and standard deviation $f_C, \sigma^2_C, \sigma_C$ 
- compare to individual stocks variance as weekly and annualized absolute quantities

In [67]:
# Python Mini Project Part 1
# Numpy for Linear Algebra 
# Yfinance for Data 

import numpy as np 
import yfinance as yf 
import matplotlib.pyplot as plt

Extract weekly closing price data for $n + 1$ weeks using yfinance api call 

In [68]:
# p = 9 stocks
stocks = ["AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "TSLA", "META", "BRK-B", "UNH"];

#Time Frame 27 weeks (1 additional for the returns calculation), n + 1 = 27
start_date = "2025-05-26"
end_date = "2025-11-28"

# Download the weekly data and grab closing prices
data = yf.download(stocks, start= start_date, end = end_date, interval = "1wk")
closing_prices = data[["Close"]]

[*********************100%***********************]  9 of 9 completed


Compute weekly total returns matrix: $(price_n - price_{n-1}) / price_{n-1}$ and take transpose for desired dimensions

In [69]:
#Calculate Returns
returns = closing_prices.pct_change().dropna()

# Convert to p x n Matrix 
returns_matrix = returns.T.to_numpy()
print(returns_matrix.shape)

(9, 26)


Compute weekly excess returns matrix: $ExcessReturns = TotalReturns - RiskFreeRate$

In [70]:
# Calculate weekly excess returns matrix
risk_free_rate = 0.0409 # 10-yr treasury bonds 
weekly_RFR = risk_free_rate/52 # 

# Excess Returns Formula
excess_returns_matrix = returns_matrix - weekly_RFR 

Compute mean vector $\mu$ and de-meaned weekly excess returns matrix: $Y = ExcessReturnsMatrix - \mu$

In [71]:
# axis=1 takes mean of each row and creates a column vector of each stocks mean 
# keepdims=True enable matrix subtraction by copying each collumn to match the matrix being worked on  
excess_returns_mean_vector = excess_returns_matrix.mean(axis=1, keepdims= True)

# Creating the p x n de-meaned excess returns matrix Y
Y = excess_returns_matrix - excess_returns_mean_vector

Compute weekly sample Covariance Matrix: $ S = YY^T/n$ 

In [72]:
# @ is the python symbol for matrix mult
S = (Y @ Y.T) / 26
print(S.shape)

(9, 9)


Compute global-minimum-variance portfolio holding vector: $h_C = S^{-1}e / e^TSe$ where $e = (1,1,...,1)$

In [73]:
# Compute Global-minimum-Variance Pfolio C 
e = np.ones(9, dtype=float) # characteristic of C 
S_inverse = np.linalg.inv(S) # Inverse Covariance Matrix

# Holding pfolio for C 
holdings_C = (S_inverse @ e) / (e.T @ S_inverse @ e)

print("Total Holdings Vector Sum (fully invested)", sum(holdings_C))
print("GMV Pfolio C Holdings Vector \n", holdings_C)

Total Holdings Vector Sum (fully invested) 1.0
GMV Pfolio C Holdings Vector 
 [ 0.01934604 -0.07155861  0.66082314  0.09737242  0.06003337  0.02255709
  0.23507029  0.01656706 -0.0402108 ]


Compute GMV portfolio variance $\sigma^2_C = h^T_C S h_C$

In [74]:
# Calculate the variance 
Variance_C = (holdings_C.T @ S @ holdings_C)
annual_Variance_C = Variance_C * 52 
print("Pfolio C Variance (weekly): ", Variance_C)
print("Pfolio C Variance (annual): ", annual_Variance_C)

Pfolio C Variance (weekly):  0.00012197756687016413
Pfolio C Variance (annual):  0.006342833477248535


Compute GMV portfolio standard deviation: $\sigma = \sqrt(h^T_C S h_C)$

In [75]:
# Compute Standard Deviation 
std_pfolo_C = np.sqrt(Variance_C)
annualized_std_pfolio_C = std_pfolo_C * np.sqrt(52)

print("Portfolio C Standard Deviation (weekly): ", std_pfolo_C)
print("Portfolio C Standard Deviation (annual): ", annualized_std_pfolio_C)

Portfolio C Standard Deviation (weekly):  0.011044345470428029
Portfolio C Standard Deviation (annual):  0.07964190779513342


Compute expected excess returns: $f_C = E[r_C] = h^T_C  \mu$

In [76]:
# Compute expected excess returns
expected_excess_returns_C = holdings_C.T @ excess_returns_mean_vector
annualized_expected_excess_returns_C = expected_excess_returns_C * 52

print("Expected Excess Returns for Portfolio C (weekly): ",expected_excess_returns_C)
print("Expected Excess Returns for Portfolio C (annual): ", annualized_expected_excess_returns_C)

Expected Excess Returns for Portfolio C (weekly):  [0.00481942]
Expected Excess Returns for Portfolio C (annual):  [0.25061004]


Compute beta with respect to portfolio: $\beta_C = Sh_C / \sigma^2 = e$

In [77]:
# Calculate Beta WRT Pfolio C
Beta_C = (S @ holdings_C) / Variance_C
print("Pfolio C Beta: ", Beta_C)

Pfolio C Beta:  [1. 1. 1. 1. 1. 1. 1. 1. 1.]


The individual stocks variance are contained on the diagonal of $S$ by the fact that the elements $S_{ij}$ are the covariances between stocks $i$ and $j$. Then for the diagonal elements $i=j$ the covariance is with respect to the same stock implying $Cov(S_i,S_j) = Cov(S_i,S_i) = Var(S_i)$ 

In [78]:
#Compute Variance for indivdual stocks 
# store the diagonal elements in array 
individual_variance = np.diag(S)
annual_individual_variance = individual_variance * 52

# loop through the array and print that stocks variance
for i in range(len(individual_variance)):
    print(str(stocks[i]) + " Variance (weekly): ", individual_variance[i])
    print(str(stocks[i]) + " Variance (annual): ", annual_individual_variance[i])
    print("\n")

AAPL Variance (weekly):  0.0013643912731610465
AAPL Variance (annual):  0.07094834620437442


MSFT Variance (weekly):  0.0014156828735306436
MSFT Variance (annual):  0.07361550942359346


AMZN Variance (weekly):  0.0003416525929019789
AMZN Variance (annual):  0.017765934830902903


NVDA Variance (weekly):  0.0015791952389190968
NVDA Variance (annual):  0.08211815242379303


GOOGL Variance (weekly):  0.001962332599477652
GOOGL Variance (annual):  0.10204129517283791


TSLA Variance (weekly):  0.0006078288510036734
TSLA Variance (annual):  0.03160710025219102


META Variance (weekly):  0.0015963000142393
META Variance (annual):  0.0830076007404436


BRK-B Variance (weekly):  0.00379327286058948
BRK-B Variance (annual):  0.19725018875065298


UNH Variance (weekly):  0.0040867593776678195
UNH Variance (annual):  0.21251148763872663




### Results
This portfolio seems very reasonable to invest in if the investors preference is to minimize their variance. Comparing the weekly variance of the portfolio C $\approx$ 0.000122 to lowest weekly variance among our individual stocks (AMZN) $\approx$ 0.000342 there is a considerable increase in the variance that helps support our conclusion. 